**Solvation fix, 19 November 2026:** This version corrects an m6A methyl-hydrogen name mismatch when reloading the repaired structure. Previously completed cases are reused; failed cases without a saved case result can be retried by rerunning the batch. Saved unconverged results are preserved and are not automatically retried.

# Docking pose 4

This copy writes to `MyDrive/PABPC1_MMGBSA_pose_04`. Use the three notebooks in this same pose folder together. The original notebooks use `MyDrive/PABPC1_MMGBSA`. This copy can reuse the original system’s workbooks; select this pose’s own PDB when prompted.

The input checker requires RNA chain E containing 30 adenines and protein chain A with 400 residues whose first residue IDs are 8, 9, and 11; it rebuilds the missing Met10. A different chain layout, sequence, or protein model needs adaptation before use. The repaired pose and calculated scores remain exploratory.

# Single-frame GBn1 scoring

Run after notebook 1 and, if desired, notebook 2. This notebook may run on a CPU runtime. It uses direct Amber `sander` because native AmberTools 26 `MMPBSA.py` does not expose GB `intdiel=2` in its `&gb` input. The equivalent direct scoring calculation was checked against native MMPBSA.py at dielectric 1. Scores omit conformational entropy and are exploratory.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/PABPC1_MMGBSA_pose_04')  # Change consistently in all three notebooks if desired.
INPUTS = ROOT / 'inputs'
INPUTS.mkdir(parents=True, exist_ok=True)
print('Shared project folder:', ROOT)


In [ ]:
required = ['Muller_benchmark.xlsx', 'Muller_Position_Top_Two_Chemical_Modification_Library.xlsx', 'temp_complex.pdb']
missing = [name for name in required if not (INPUTS / name).is_file()]
if missing: raise FileNotFoundError('Provide the inputs and run a minimization notebook first: ' + ', '.join(missing))
if not (ROOT / 'template.pdb').is_file():
    raise FileNotFoundError('Run a minimization notebook to create the human template first.')


In [ ]:
import os, io, tarfile, urllib.request, subprocess, zipfile, base64
from pathlib import Path

RUNTIME_DIR = Path('/content/pabpc1_pose04_runtime')  # Scratch space; Drive contains persistent results.
ENV_DIR = RUNTIME_DIR / 'environment'
ASSETS = RUNTIME_DIR / 'assets'
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
# The archive contains code, 23 modXNA libraries, the modXNA frcmod file,
# and the ModeRNA source and licenses. Amber ff14SB/OL3 come from AmberTools.
assets_b64 = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(assets_b64))) as z:
    z.extractall(ASSETS)

mamba = RUNTIME_DIR / 'bin' / 'micromamba'
if not mamba.exists():
    mamba.parent.mkdir(parents=True, exist_ok=True)
    archive = RUNTIME_DIR / 'micromamba.tar.bz2'
    urllib.request.urlretrieve('https://micro.mamba.pm/api/micromamba/linux-64/latest', archive)
    with tarfile.open(archive, 'r:bz2') as t:
        member = t.extractfile('bin/micromamba')
        if member is None: raise RuntimeError('micromamba archive is missing its executable')
        mamba.write_bytes(member.read())
    mamba.chmod(0o755)
if not (ENV_DIR / 'installation_complete').exists():
    (ENV_DIR / 'conda-meta').mkdir(parents=True, exist_ok=True)
    menv = dict(os.environ, MAMBA_ROOT_PREFIX=str(ENV_DIR),
                CONDA_PKGS_DIRS=str(RUNTIME_DIR / 'packages'), XDG_CACHE_HOME=str(RUNTIME_DIR / 'cache'))
    command = [str(mamba), 'install', '-y', '-p', str(ENV_DIR), '-c', 'conda-forge',
               'python=3.11', 'ambertools=26.0', 'openmm=8.2.0', 'cuda-version=12.4',
               'pdbfixer=1.12', 'parmed=4.3.1', 'biopython=1.88', 'scipy=1.13.1',
               'numpy=1.26.4', 'openpyxl=3.1.5']
    subprocess.run(command, env=menv, check=True)
    (ENV_DIR / 'installation_complete').write_text('OK')
worker_env = dict(os.environ)
worker_env['PATH'] = str(ENV_DIR / 'bin') + os.pathsep + worker_env.get('PATH', '')
worker_env['AMBERHOME'] = str(ENV_DIR)
worker_env['PYTHONPATH'] = str(ASSETS / 'src') + os.pathsep + str(ASSETS / 'rna')

def worker(source):
    p = subprocess.Popen([str(ENV_DIR / 'bin/python'), '-u', '-c', source], env=worker_env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout: print(line, end='')
    code = p.wait()
    if code: raise RuntimeError('Worker failed with exit status ' + str(code))
print('Environment ready; assets are bundled. First installation can take several minutes.')


In [ ]:
# CPU scoring; repeat this cell to pick up cases completed later.
# Nonconverged or failed cases are deliberately left blank in the reports.
if not (ROOT / 'index.csv').is_file(): raise FileNotFoundError('Run a minimization notebook first.')
worker(f"""from report import score_batch
print(score_batch({str(ROOT)!r},'/content/pabpc1_pose04_score_cache'))
""")
print('Reports:', ROOT / 'benchmark_scores.csv', 'and', ROOT / 'matched_variant_scores.csv')


## Interpreting results

A more negative variant-minus-parent score indicates a more favorable **modeled score** under this one-structure protocol. Compare each candidate with its own control first. Absolute scores, small differences, and differences between unrelated parents are not calibrated binding free energies. Gaps in the CSV mean missing, failed or unconverged cases, not zero binding.

The revised 34-sequence workbook has no sequence-specific measured binding enrichments. It cannot establish a correlation with experiment until measured values are obtained. Müller’s measurements used longer 50-nt UTRs, and that study’s position −1 convention can differ from the annotated junction used here. The full calculation must be treated as exploratory until validated on an appropriate experiment.

Check `failures/` and each `minimization.json` on Drive. Rerunning skips completed cases and preserves existing unconverged results. Files are gzip-compressed; the scoring notebook decompresses them automatically. The first completed-case runtime is only a rough guide, not a completion guarantee.
